# 05. Project Foundation, Source Inventory, and Schema

    Objective: prove that the anime recommender has a valid domain, valid sources, a reproducible first dataset build, and a schema that supports representation, clustering, recommendation, and graph analysis.

## Project Proposal

    *Domain.* Anime discovery and recommendation.

    *Problem statement.* Anime catalogs are large, sparse, and fragmented across platforms. A viewer may know a few shows they like but still struggle to discover similar titles, hidden related works, or thematic clusters. This project builds a reproducible anime catalog that combines MyAnimeList metadata, AniDB tags, MAL recommendation edges, MAL relation edges, and optional user ratings.

    *Product question.* Given an anime or a viewer preference profile, which anime are similar, which latent segment do they belong to, and which titles should be recommended next?

    *Course fit.* The dataset supports a catalog layer, feature layer, interaction layer, graph layer, and reproducible pipeline. It is not a single supervised model project.

In [1]:
from pathlib import Path
from collections import Counter
import json
import math
import re

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 120)
pd.set_option("display.max_colwidth", 160)

BASE_DIR = Path.cwd().resolve()
if BASE_DIR.name == "notebooks":
    BASE_DIR = BASE_DIR.parent

DATA_DIR = BASE_DIR / "data"
PROCESSED_DIR = DATA_DIR / "processed"
BUILD_DIR = DATA_DIR / "build"
PLOT_DIR = BASE_DIR / "artifacts" / "plots"

ANIME_PATH = PROCESSED_DIR / "anime_dataset.csv"
RATINGS_PATH = PROCESSED_DIR / "current_user_ratings.csv"

def split_pipe(value):
    if pd.isna(value) or str(value).strip() == "":
        return []
    return [part.strip() for part in str(value).split("|") if part.strip()]

def count_pipe(value):
    return len(split_pipe(value))

def explode_pipe_counts(series):
    counts = Counter()
    for value in series.dropna():
        counts.update(split_pipe(value))
    return pd.DataFrame(counts.most_common(), columns=["value", "count"])

def parse_duration_minutes(value):
    if pd.isna(value) or str(value).strip() == "":
        return np.nan
    if isinstance(value, (int, float)) and not pd.isna(value):
        return float(value) if float(value) > 0 else np.nan

    text = str(value).strip().lower()
    if re.fullmatch(r"\d+(?:\.\d+)?", text):
        numeric = float(text)
        return numeric if numeric > 0 else np.nan

    hours = re.search(r"(\d+(?:\.\d+)?)\s*(?:hr|hour)", text)
    minutes = re.search(r"(\d+(?:\.\d+)?)\s*min", text)
    seconds = re.search(r"(\d+(?:\.\d+)?)\s*sec", text)
    total = 0.0
    if hours:
        total += float(hours.group(1)) * 60
    if minutes:
        total += float(minutes.group(1))
    if seconds:
        total += float(seconds.group(1)) / 60
    return total if total > 0 else np.nan

def infer_season(month):
    if pd.isna(month):
        return np.nan
    month = int(month)
    if month in [1, 2, 3]:
        return "winter"
    if month in [4, 5, 6]:
        return "spring"
    if month in [7, 8, 9]:
        return "summer"
    if month in [10, 11, 12]:
        return "fall"
    return np.nan

def add_bar_labels(ax, fmt="{:.0f}"):
    for patch in ax.patches:
        width = patch.get_width()
        ax.text(width, patch.get_y() + patch.get_height() / 2, " " + fmt.format(width), va="center", fontsize=9)

def save_current_plot(path):
    path.parent.mkdir(parents=True, exist_ok=True)
    plt.tight_layout()
    plt.savefig(path, dpi=160, bbox_inches="tight")

df = pd.read_csv(ANIME_PATH)
summary_path = BUILD_DIR / "dataset_build_summary.json"
summary = json.loads(summary_path.read_text(encoding="utf-8")) if summary_path.exists() else {}
print(f"Anime rows: {len(df):,}")
print(f"Columns: {df.shape[1]:,}")
display(df.head())
display(summary)

Anime rows: 15,492
Columns: 35


,mal_id,anilist_id,anidb_id,url,image_url,title,title_english,type,source,episodes,duration,total_watch_minutes,status,rating,score,scored_by,rank,popularity,members,favorites,aired_year,aired_month,season,genres,tags,tag_weights,explicit_tags,explicit_tag_weights,demographics,studios,characters,voice_actors,relations,recommendations,hentai_rank
0,1,1.0,23.0,https://myanimelist.net/anime/1/Cowboy_Bebop,https://cdn.myanimelist.net/images/anime/4/19644l.jpg,Cowboy Bebop,Cowboy Bebop,TV,Original,26,24.0,624.0,Finished Airing,R - 17+ (violence & profanity),8.75,1065859,48,41,2065698,90022,1998.0,4.0,spring,Action|Adventure|Drama|Sci-Fi,Space|Crime|Episodic|Ensemble Cast|Primarily Adult Cast|Philosophy|Tragedy|Travel|Found Family|Noir|Anti-Hero|Male Protagonist|Cyberpunk|Guns|Female Protago...,Space:94|Crime:92|Episodic:89|Ensemble Cast:86|Primarily Adult Cast:85|Philosophy:84|Tragedy:83|Travel:82|Found Family:82|Noir:80|Anti-Hero:79|Male Protagon...,NaN,NaN,Seinen,Sunrise,"3:Main:Black, Jet|1:Main:Spiegel, Spike|2:Main:Valentine, Faye|16:Main:Wong Hau Pepelu Tivrusky IV, Edward|131427:Supporting:Actress|131428:Supporting:Alisa...","357:Ishizuka, Unshou|11:Yamadera, Kouichi|14:Hayashibara, Megumi|658:Tada, Aoi|28:Horie, Yui|155:Doi, Mika|800:Tsuji, Shinpachi|276:Kojima, Sachiko|8016:Ich...",5:Side Story|17205:Side Story|4037:Summary,205:689|6:317|40052:214|20057:177|889:159|2251:99|56038:88|400:86|4087:50|467:46|52093:40|42310:38|1412:34|30:34|28977:34|2025:32|24439:29|567:27|918:24|474...,NaN
1,5,5.0,219.0,https://myanimelist.net/anime/5/Cowboy_Bebop__Tengoku_no_Tobira,https://cdn.myanimelist.net/images/anime/1439/93480l.jpg,Cowboy Bebop: Tengoku no Tobira,Cowboy Bebop: The Movie,Movie,Original,1,115.0,115.0,Finished Airing,R - 17+ (violence & profanity),8.38,232750,240,659,413272,1793,2001.0,9.0,summer,Action|Drama|Mystery|Sci-Fi,Terrorism|Primarily Adult Cast|Philosophy|Martial Arts|Crime|Urban|Space|Noir|Amnesia|Cyberpunk|Anti-Hero|Ensemble Cast|Guns|Male Protagonist|Tomboy|Foreign...,Terrorism:98|Primarily Adult Cast:86|Philosophy:81|Martial Arts:80|Crime:80|Urban:79|Space:76|Noir:76|Amnesia:75|Cyberpunk:73|Anti-Hero:73|Ensemble Cast:71|...,NaN,NaN,Seinen,Bones,"3:Main:Black, Jet|8435:Main:Ovilo, Electra|1:Main:Spiegel, Spike|2:Main:Valentine, Faye|4035:Main:Volaju, Vincent|16:Main:Wong Hau Pepelu Tivrusky IV, Edwar...","357:Ishizuka, Unshou|1019:Kobayashi, Ai|11:Yamadera, Kouichi|14:Hayashibara, Megumi|450:Isobe, Tsutomu|658:Tada, Aoi|18:Shibata, Hidekatsu|634:Akimoto, Yous...",1:Parent Story,43:26|4106:22|20057:17|1095:14|15335:6|1430:4|40858:4|2013:4|6:3|21339:2|122:2|522:2|52093:2|9135:1|23279:1|1796:1|393:1|570:1|1226:1|28211:1|1023:1|4639:1|...,NaN
2,6,6.0,53.0,https://myanimelist.net/anime/6/Trigun,https://cdn.myanimelist.net/images/anime/1130/120002l.jpg,Trigun,Trigun,TV,Manga,26,24.0,624.0,Finished Airing,PG-13 - Teens 13 or older,8.22,402576,411,266,837709,17761,1998.0,4.0,spring,Action|Adventure|Comedy|Drama|Sci-Fi,Guns|Fugitive|Male Protagonist|Philosophy|Primarily Adult Cast|Cowboys|Tragedy|Post-Apocalyptic|Crime|Desert|Travel|Steampunk|Twins|Episodic|Aliens|Space|Sl...,Guns:92|Fugitive:92|Male Protagonist:82|Philosophy:81|Primarily Adult Cast:79|Cowboys:79|Tragedy:77|Post-Apocalyptic:77|Crime:76|Desert:76|Travel:76|Steampu...,NaN,NaN,Shounen,Madhouse,"713:Main:Stryfe, Meryl|714:Main:Thompson, Milly|162:Main:Vash the Stampede|722:Main:Wolfwood, Nicholas D.|129896:Supporting:Aura Cayzen, Marianne|223161:Sup...","354:Tsuru, Hiromi|3:Yukino, Satsuki|173:Onosaka, Masaya|344:Miyata, Kouki|137:Hayami, Show|1111:Sakuma, Rei|798:Yanada, Kiyoyuki|77760:Suzuki, Shousei|6504:...",4106:Side Story|52093:Alternative Version,1:317|121:105|27:41|45:37|267:37|411:32|24439:32|297:32|777:29|25:27|400:26|48414:24|54863:22|889:20|34451:18|38903:16|569:15|205:14|650:14|5114:14|68:12|42...,NaN
3,7,7.0,96.0,https://myanimelist.net/anime/7/Witch_Hunter_Robin,https://cdn.myanimelist.net/images/anime/10/19969l.jpg,Witch Hunter Robin,Witch H

{'updated_at': '2026-07-04T21:04:03',
 'rows': 15527,
 'raw_jikan_entries': 30186,
 'raw_jikan_character_entries': 17297,
 'raw_anilist_entries': 17274,
 'raw_anidb_entries': 15842,
 'anilist_cache_used': 'data/raw_sources/anilist/anilist_media_cache.json',
 'anidb_cache_used': 'data/raw_sources/anidb/anidb_metadata_cache.json',
 'anilist_rows': 14620,
 'anidb_rows': 13545,
 'discrepancies': 22722,
 'anilist_split_episode_repairs': 52,
 'dropped_no_episode_count': 6,
 'recap_like_rows_detected': 721,
 'dropped_recap_like_rows': 45,
 'kept_recap_like_rows': 675,
 'dropped_sparse_low_signal_rows': 1165,
 'inherited_or_inferred_fields': {'genres': 72,
  'tags': 451,
  'tag_weights': 451,
  'explicit_tags': 371,
  'explicit_tag_weights': 371,
  'demographics': 856,
  'rating': 11,
  'studios': 175,
  'demographics_inferred': 469,
  'rating_inferred': 20},
 'note': 'Genres/tags primarily come from AniList; MAL genres/themes fill only blank AniList labels, with MAL Erotica/Hentai and AniDB L

## Source Inventory

    Sources used:

    - Jikan API for MAL anime metadata. Jikan is an unofficial public API for MyAnimeList pages. It provides catalog fields such as title, type, score, status, dates, genres, themes, demographics, relations, recommendations, and external links.
    - AniDB HTTP API for AniDB metadata when needed, especially tags and episode counts for currently airing titles.
    - Shoko Anime HTTP XML cache as a cached AniDB XML source. This reduces live AniDB API calls and lowers ban risk.
    - Public MAL ID cache from `purarue/mal-id-cache` for candidate MAL IDs.
    - Current public MAL-list collector for interactions, stored as `current_user_ratings.csv` and `current_user_profile_features.csv`.

    Highest-risk source: AniDB live HTTP API, because it can ban clients when hit too aggressively. Mitigation: use Shoko cache first, use live calls only when needed, keep retry logs, and store deleted/missing AniDB IDs as permanent AniDB skips.

In [2]:
source_inventory = pd.DataFrame([
    {"source": "Jikan API", "role": "MAL catalog metadata", "format": "JSON", "local_artifact": "data/processed/anime_dataset.csv", "access": "public API, rate-limited"},
    {"source": "AniDB HTTP API", "role": "episode counts and tag metadata", "format": "XML", "local_artifact": "data/caches/anidb_metadata_cache.json", "access": "public API with strict usage limits"},
    {"source": "Shoko Anime_HTTP.zip", "role": "cached AniDB XML seed", "format": "ZIP of XML files", "local_artifact": "data/caches/anidb_metadata_cache.json", "access": "public cache"},
    {"source": "purarue MAL ID cache", "role": "candidate MAL IDs", "format": "JSON", "local_artifact": "data/raw/mal_candidate_ids.json", "access": "public GitHub data"},
    {"source": "Current public MAL-list collector", "role": "user-item ratings interaction layer", "format": "CSV", "local_artifact": "data/processed/current_user_ratings.csv", "access": "Public MAL profiles/lists via official API and public pages"},
])
source_inventory

,source,role,format,local_artifact,access
0,Jikan API,MAL catalog metadata,JSON,data/processed/anime_dataset.csv,"public API, rate-limited"
1,AniDB HTTP API,episode counts and tag metadata,XML,data/caches/anidb_metadata_cache.json,public API with strict usage limits
2,Shoko Anime_HTTP.zip,cached AniDB XML seed,ZIP of XML files,data/caches/anidb_metadata_cache.json,public cache
3,purarue MAL ID cache,candidate MAL IDs,JSON,data/raw/mal_candidate_ids.json,public GitHub data
4,Current public MAL-list collector,user-item ratings interaction layer,CSV,data/processed/current_user_ratings.csv,Public MAL profiles/lists via official API and public pages


## Schema Draft

    Main grain: one row per MAL anime entry in `anime_dataset`.

    Key fields:

    - `mal_id`: primary catalog key.
    - `anidb_id`: optional external key to AniDB.
    - `relations`: MAL relation graph edges stored as `relation_type:target_mal_id`.
    - `recommendations`: MAL recommendation edges stored as `target_mal_id:votes`.
    - `tags`, `tag_weights`, `explicit_tags`, `explicit_tag_weights`: feature fields from MAL and AniDB.
    - `current_user_ratings.csv`: current user-item interaction layer keyed by `animeID`, aligned to `mal_id`, with score and list status.

In [3]:
schema = pd.DataFrame({
    "column": df.columns,
    "dtype": df.dtypes.astype(str).values,
    "non_null": df.notna().sum().values,
    "missing": df.isna().sum().values,
    "unique_values": [df[col].nunique(dropna=True) for col in df.columns],
})
schema

,column,dtype,non_null,missing,unique_values
0,mal_id,int64,15492,0,15492
1,anilist_id,float64,14598,894,14598
2,anidb_id,float64,13527,1965,11808
3,url,object,15492,0,15492
4,image_url,object,15492,0,15491
5,title,object,15492,0,15492
6,title_english,object,9270,6222,9162
7,type,object,15492,0,5
8,source,object,15138,354,17
9,episodes,int64,15492,0,210


## Data Dictionary Draft

    This draft gives meanings and quality notes for core fields. It is intentionally more than a column-name repeat.

In [4]:
data_dictionary = pd.DataFrame([
    {"field": "mal_id", "meaning": "MyAnimeList anime identifier", "type": "integer", "quality_note": "Primary key for catalog layer"},
    {"field": "anidb_id", "meaning": "AniDB anime identifier from MAL external links or fallback", "type": "integer/null", "quality_note": "May be missing or deleted in AniDB"},
    {"field": "type", "meaning": "MAL release type", "type": "category", "quality_note": "Allowed types include TV, Movie, OVA, ONA, Special, TV Special"},
    {"field": "score", "meaning": "MAL community score", "type": "float", "quality_note": "Rows without score are filtered out"},
    {"field": "episodes", "meaning": "Episode count", "type": "float/integer", "quality_note": "Airing shows may need AniDB refresh"},
    {"field": "duration", "meaning": "Episode duration in minutes", "type": "numeric", "quality_note": "Converted from MAL duration text during dataset improvements"},
    {"field": "total_watch_minutes", "meaning": "episodes multiplied by duration minutes", "type": "numeric", "quality_note": "Null when either episode count or duration is unavailable"},
    {"field": "genres", "meaning": "MAL genre list", "type": "pipe-delimited text", "quality_note": "Used for categorical features"},
    {"field": "tags", "meaning": "MAL themes plus AniDB non-explicit tags", "type": "pipe-delimited text", "quality_note": "AniDB weights preserve importance"},
    {"field": "explicit_tags", "meaning": "Adult/content-warning tags separated from normal tags", "type": "pipe-delimited text", "quality_note": "Missing only matters for explicit-rated rows"},
    {"field": "demographics", "meaning": "MAL or AniDB audience classification", "type": "pipe-delimited category", "quality_note": "AniDB can fill MAL empty demographics"},
    {"field": "relations", "meaning": "MAL relation edges", "type": "pipe-delimited graph edge list", "quality_note": "Supports relation graph"},
    {"field": "recommendations", "meaning": "MAL recommendation edges with votes", "type": "pipe-delimited weighted edge list", "quality_note": "Supports recommendation graph"},
])
data_dictionary

,field,meaning,type,quality_note
0,mal_id,MyAnimeList anime identifier,integer,Primary key for catalog layer
1,anidb_id,AniDB anime identifier from MAL external links or fallback,integer/null,May be missing or deleted in AniDB
2,type,MAL release type,category,"Allowed types include TV, Movie, OVA, ONA, Special, TV Special"
3,score,MAL community score,float,Rows without score are filtered out
4,episodes,Episode count,float/integer,Airing shows may need AniDB refresh
5,duration,Episode duration in minutes,numeric,Converted from MAL duration text during dataset improvements
6,total_watch_minutes,episodes multiplied by duration minutes,numeric,Null when either episode count or duration is unavailable
7,genres,MAL genre list,pipe-delimited text,Used for categorical features
8,tags,MAL themes plus AniDB non-explicit tags,pipe-delimited text,AniDB weights preserve importance
9,explicit_tags,Adult/content-warning tags separated from normal tags,pipe-delimited text,Missing only matters for explicit-rated rows


## Scale Analysis

    This table checks whether the dataset is non-trivial and whether missingness or sparsity will affect later modeling.

In [5]:
edge_cols = ["relations", "recommendations"]
scale = {
    "rows": len(df),
    "columns": df.shape[1],
    "unique_mal_id": df["mal_id"].nunique(),
    "unique_anidb_id": df["anidb_id"].nunique(dropna=True),
    "memory_mb": round(df.memory_usage(deep=True).sum() / (1024 ** 2), 2),
    "rows_with_tags": int(df["tags"].notna().sum()),
    "rows_with_recommendations": int(df["recommendations"].notna().sum()),
    "rows_with_relations": int(df["relations"].notna().sum()),
}
display(pd.DataFrame([scale]).T.rename(columns={0: "value"}))
missing = schema[["column", "missing"]].copy()
missing["missing_pct"] = missing["missing"] / len(df) * 100
display(missing.sort_values("missing_pct", ascending=False).head(15))

,value
rows,15492.00
columns,35.00
unique_mal_id,15492.00
unique_anidb_id,11808.00
memory_mb,32.15
rows_with_tags,15371.00
rows_with_recommendations,12012.00
rows_with_relations,10215.00


,column,missing,missing_pct
34,hentai_rank,13956,90.085205
26,explicit_tags,12800,82.623289
27,explicit_tag_weights,12800,82.623289
6,title_english,6222,40.162665
32,relations,5277,34.062742
28,demographics,3541,22.856958
33,recommendations,3480,22.463207
31,voice_actors,3226,20.823651
2,anidb_id,1965,12.683966
30,characters,1699,10.966951


## Processed Dataset V1 and Reproducibility

    Processed artifacts:

    - `data/processed/anime_dataset.csv`
    - `data/processed/anime_dataset.json`
    - `data/processed/current_user_ratings.csv`
    - `data/caches/anidb_metadata_cache.json`
    - build logs and retry registries under `data/build/` and `logs/`

    Reproducible command path:

    - Raw source gathering notebook: run `notebooks/01_gather_raw_sources.ipynb`.
    - Dataset build notebook: run `notebooks/02_build_anime_dataset.ipynb`.
    - Dataset audit notebook: run `notebooks/03_catalog_eda.ipynb`.
    - Ratings collector notebook: run `notebooks/00_collect_current_user_ratings.ipynb`.
    - Scripted feature build: run `python src/05_build_catalog_features.py`.

    The ingestion logs, checkpoint file, failed request registry, invalid type registry, and permanent skip registry provide evidence that the build is reproducible and auditable rather than manually assembled.

## Ethics and Access Note

    The catalog data comes from public MAL/Jikan, AniList, and AniDB/Shoko metadata. The interaction layer comes from public MAL profile/list collection and stores hashed/de-identified user ids, scored catalog-matched anime, list status, and derived profile features.

    The project does not scrape private profiles or bypass access controls. Personal-data risk is low but not zero because ratings are behavioral traces. We reduce risk by using processed user IDs only, not redistributing raw personal exports, and focusing on aggregate models rather than identifying users.

## NOTES

    - Exact product question: anime discovery using content similarity, latent clusters, recommendation edges, and graph centrality.
    - Grain: one row per MAL anime; ratings are one row per user-anime interaction.
    - Highest-risk source: AniDB live API; mitigated with cache-first strategy and permanent skip rules.
    - Later layers: tags/text/numeric features, dimensionality reduction, K-means/DBSCAN clustering, recommendation graph, ratings matrix.
    - Limitations: MAL and AniDB disagree on specials; some AniDB IDs are deleted; recommendations are platform-generated and popularity-biased.